# Before training: feature KNN consensus

Standalone Kaggle experiment using competition data and scratch weights. Validation is an untouched partition of noisy training labels. Before methods freeze labels/subset before initializing the final classifier. During methods start from the original noisy labels.

In [ ]:
"""Competition-only training. No downloads, external weights, or clean-label access."""
from dataclasses import dataclass, asdict
from pathlib import Path
import copy
import hashlib
import json
import math
import os
import pickle
import random
import shutil
import tempfile
import time
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models import resnet18
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors


@dataclass
class Config:
    method: str = 'confidence'
    data_root: str = '/kaggle/input/competitions/dl-gen-ai-2026-fii-competition-3/fii-atnn-2024-project-noisy-cifar-100'
    output: str = '/kaggle/working/experiment'
    seed: int = 42
    epochs: int = 160
    feature_epochs: int = 30
    warmup: int = 25
    ramp: int = 30
    batch: int = 128
    workers: int = 2
    lr: float = 0.1
    weight_decay: float = 5e-4
    val_fraction: float = 0.1
    retention: float = 0.8
    loss_decay: float = 0.8
    prob_decay: float = 0.8
    ema_decay: float = 0.99
    confidence: float = 0.9
    margin: float = 0.7
    patience: int = 5
    min_weight: float = 0.25
    correction: float = 0.5
    smoothing: float = 0.05
    neighbors: int = 50
    clusters: int = 500
    consensus: float = 0.9
    preprocess_action: str = 'relabel'
    strong: bool = False
    mixup: float = 0.0
    gradient_signal: bool = False
    tta: bool = False
    resume: bool = True
    smoke: bool = False
    submission: bool = False
    submission_root: str = ''  # Notebook sets /kaggle/working/submission.csv.
    template: str = ''
    confirm_positional_ids: bool = False
    freeze_noise_epoch: int = 0  # One-based fixed epoch; 0 keeps reversible updates.
    recalibrate_bn: bool = False  # Training images only; extra forward passes.


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_training(root):
    root = Path(root)
    path = root / 'cifar-100-python/train'
    if not path.is_file():
        raise FileNotFoundError(f'Expected competition train file: {path}; set data_root explicitly.')
    # Pickle necessarily deserializes the archive; only the image field is accessed.
    with path.open('rb') as f:
        images = pickle.load(f, encoding='bytes')[b'data']
    with np.load(root / 'CIFAR-100-noisy.npz', allow_pickle=False) as archive:
        labels = archive['noisy_label'].astype(np.int64)
    if images.ndim != 2 or images.shape[1] != 3072 or images.dtype != np.uint8:
        raise ValueError('Expected uint8 competition CIFAR images, shape (N,3072).')
    if labels.shape != (len(images),) or labels.min() < 0 or labels.max() >= 100:
        raise ValueError('Noisy labels must align with images and lie in [0,99].')
    return images.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1), labels


def split_indices(n, cfg):
    order = np.random.default_rng(cfg.seed).permutation(n)
    cut = int((1 - cfg.val_fraction) * n)
    train, val = order[:cut], order[cut:]
    if cfg.smoke:
        train, val = train[:256], val[:128]
    assert len(train) and len(val) and not np.intersect1d(train, val).size
    return train, val


class Images(Dataset):
    def __init__(self, images, labels, ids, augment=False, strong=False):
        self.images, self.labels = images, labels
        self.ids = np.asarray(ids)
        transforms = [T.ToPILImage()]
        if augment:
            transforms += [T.RandomCrop(32, padding=4), T.RandomHorizontalFlip()]
            if strong:
                transforms += [T.RandAugment(num_ops=2, magnitude=5)]
        self.transform = T.Compose(transforms + [T.ToTensor(), T.Normalize((0.5,)*3, (0.25,)*3)])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        sid = int(self.ids[index])
        return self.transform(self.images[sid]), int(self.labels[sid]), sid


def loader(images, labels, ids, cfg, augment=False):
    return DataLoader(Images(images, labels, ids, augment, cfg.strong),
                      batch_size=cfg.batch, shuffle=augment, num_workers=cfg.workers,
                      pin_memory=torch.cuda.is_available(), persistent_workers=cfg.workers > 0)


def new_model(device):
    model = resnet18(weights=None, num_classes=100)
    model.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(device)


@torch.inference_mode()
def inspect_training(model, batches, n, device, embeddings=False):
    model.eval()
    probs, losses = torch.zeros(n, 100), torch.zeros(n)
    features = torch.zeros(n, 512) if embeddings else None
    captured = []
    handle = model.fc.register_forward_pre_hook(lambda module, args: captured.append(args[0])) if embeddings else None
    try:
        for x, y, ids in batches:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device.type, enabled=device.type == 'cuda'):
                z = model(x)
            probs[ids] = z.float().softmax(1).cpu()
            losses[ids] = F.cross_entropy(z.float(), y, reduction='none').cpu()
            if embeddings:
                features[ids] = F.normalize(captured.pop().float(), dim=1).cpu()
    finally:
        if handle is not None:
            handle.remove()
    return probs, losses, features


@torch.inference_mode()
def evaluate(model, batches, device, tta=False):
    model.eval()
    correct, loss, count = 0, 0., 0
    for x, y, _ in batches:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device.type, enabled=device.type == 'cuda'):
            z = model(x)
            if tta:
                p = (z.float().softmax(1) + model(x.flip(-1)).float().softmax(1))/2
        if tta:
            correct += (p.argmax(1) == y).sum().item()
            loss += F.nll_loss(p.clamp_min(1e-12).log(), y, reduction='sum').item()
        else:
            correct += (z.argmax(1) == y).sum().item()
            loss += F.cross_entropy(z.float(), y, reduction='sum').item()
        count += len(y)
    return 100 * correct / count, loss / count


@torch.no_grad()
def calibrate_batchnorm(model, training_batches, device):
    """Recompute BN buffers on unmixed TRAIN images, without label/parameter updates."""
    batchnorms = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)
                  and m.track_running_stats]
    if not batchnorms:
        return
    modes = {m: m.training for m in model.modules()}
    momenta = {m: m.momentum for m in batchnorms}
    model.eval()
    for module in batchnorms:
        module.reset_running_stats()
        module.momentum = None
        module.train()
    try:
        for x, *_ in training_batches:
            with torch.autocast(device.type, enabled=device.type == 'cuda'):
                model(x.to(device, non_blocking=True))
    finally:
        for module, momentum in momenta.items():
            module.momentum = momentum
        for module, training in modes.items():
            module.training = training


def fixed_preprocessing(features, labels, train_ids, cfg):
    """Votes use training labels only, and exclude each sample's own vote."""
    x, y = features[train_ids].numpy(), labels[train_ids]
    votes = np.zeros((len(x), 100), dtype=np.float32)
    support = np.zeros(len(x), dtype=np.int64)
    if cfg.method == 'knn':
        k = min(cfg.neighbors, len(x) - 1)
        if k < 1:
            raise ValueError('KNN requires at least two training samples.')
        nn_index = NearestNeighbors(n_neighbors=k + 1, metric='cosine', n_jobs=-1).fit(x)
        # Chunk queries: avoid a full 45k x 45k distance matrix in RAM.
        for start in range(0, len(x), 512):
            neighbors = nn_index.kneighbors(x[start:start + 512], return_distance=False)
            for j, row in enumerate(neighbors):
                i = start + j
                row = row[row != i][:k]
                votes[i] = np.bincount(y[row], minlength=100)
                support[i] = len(row)
    else:
        clusters = MiniBatchKMeans(n_clusters=min(cfg.clusters, max(2, len(x)//8)),
                                  random_state=cfg.seed, n_init=3, batch_size=2048).fit_predict(x)
        for cluster in np.unique(clusters):
            members = np.flatnonzero(clusters == cluster)
            counts = np.bincount(y[members], minlength=100)
            votes[members] = counts - np.eye(100, dtype=np.float32)[y[members]]
            support[members] = len(members) - 1
    distributions = votes / np.maximum(support[:, None], 1)
    prediction = distributions.argmax(1)
    confidence = distributions.max(1)
    suspect = (prediction != y) & (confidence >= cfg.consensus) & (support >= 10)
    corrected = labels.copy()
    keep = np.ones(len(x), dtype=bool)
    if cfg.preprocess_action == 'relabel':
        corrected[train_ids[suspect]] = prediction[suspect]
    elif cfg.preprocess_action == 'filter':
        keep[suspect] = False
        # Retain a class if filtering would erase it completely.
        for c in np.unique(y):
            group = np.flatnonzero(y == c)
            if not keep[group].any():
                keep[group[np.argmin(confidence[group])]] = True
    else:
        raise ValueError('preprocess_action must be filter or relabel')
    return corrected, train_ids[keep], suspect


class History:
    def __init__(self, n):
        self.loss = torch.zeros(n)
        self.probs = torch.zeros(n, 100)
        self.vote = torch.full((n,), -1, dtype=torch.long)
        self.streak = torch.zeros(n, dtype=torch.long)
        self.seen = torch.zeros(n, dtype=torch.bool)

    def update(self, probs, losses, ids, cfg):
        old = self.seen[ids]
        self.loss[ids] = torch.where(old, cfg.loss_decay*self.loss[ids] + (1-cfg.loss_decay)*losses[ids], losses[ids])
        self.probs[ids] = torch.where(old[:, None], cfg.prob_decay*self.probs[ids] + (1-cfg.prob_decay)*probs[ids], probs[ids])
        confidence, winner = self.probs[ids].max(1)
        # Count consecutive confident votes, not merely consecutive argmax values.
        eligible = confidence >= cfg.confidence
        self.streak[ids] = torch.where(eligible, torch.where(winner == self.vote[ids], self.streak[ids]+1, 1), 0)
        self.vote[ids] = winner
        self.seen[ids] = True


def dynamic_targets(labels, ids, history, cfg, epoch):
    targets = F.one_hot(torch.as_tensor(labels), 100).float()
    weights = torch.ones(len(labels))
    changed = torch.zeros(len(labels), dtype=torch.bool)
    if epoch < cfg.warmup:
        return targets, weights, changed
    progress = min(1., (epoch-cfg.warmup+1)/max(1, cfg.ramp))
    if cfg.method == 'small_loss':
        # Per-class historical selection; retain a nonzero exploration contribution.
        for c in np.unique(labels[ids]):
            group = ids[labels[ids] == c]
            keep = max(1, math.ceil(len(group)*(1-progress*(1-cfg.retention))))
            ordered = group[torch.argsort(history.loss[group]).numpy()]
            weights[ordered[keep:]] = 0.05
    elif cfg.method in ('confidence', 'best'):
        p = history.probs[ids]
        confidence, winner = p.max(1)
        given = p.gather(1, torch.as_tensor(labels[ids])[:, None]).squeeze(1)
        disagreement = (confidence - given).clamp(0, 1)
        weights[ids] = (1 - progress*disagreement).clamp_min(cfg.min_weight)
        eligible = ((winner != torch.as_tensor(labels[ids])) & (confidence >= cfg.confidence)
                    & (confidence-given >= cfg.margin) & (history.streak[ids] >= cfg.patience))
        changed[ids] = eligible & (cfg.correction > 0)
        beta = cfg.correction * progress * eligible.float()
        targets[ids] = (1-beta[:, None])*targets[ids] + beta[:, None]*p
        # More smoothing for uncertain predictions, without forcing a hard correction.
        entropy = -(p*p.clamp_min(1e-9).log()).sum(1)/math.log(100)
        smoothing = cfg.smoothing * entropy
        targets[ids] = (1-smoothing[:, None])*targets[ids] + smoothing[:, None]/100
    return targets.detach(), weights.detach(), changed


def noise_targets(labels, ids, history, cfg, epoch, frozen=None):
    """Optional snapshot during training prevents late erasure of earlier suspicions."""
    if frozen is not None:
        return frozen['targets'], frozen['weights'], frozen['changed'], frozen
    targets, weights, changed = dynamic_targets(labels, ids, history, cfg, epoch)
    if cfg.freeze_noise_epoch and epoch+1 >= cfg.freeze_noise_epoch:
        frozen = dict(targets=targets.clone(), weights=weights.clone(), changed=changed.clone())
    return targets, weights, changed, frozen


def train_epoch(model, batches, optimizer, scaler, targets, weights, device, cfg, ema=None):
    model.train()
    total, count = 0., 0
    for x, _, ids in batches:
        x = x.to(device, non_blocking=True)
        t, w = targets[ids].to(device), weights[ids].to(device)
        weighted_t = t*w[:, None]
        if cfg.mixup > 0:
            lam = np.random.beta(cfg.mixup, cfg.mixup)
            order = torch.randperm(len(x), device=device)
            x = lam*x + (1-lam)*x[order]
            weighted_t = lam*weighted_t + (1-lam)*weighted_t[order]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=device.type == 'cuda'):
            z = model(x)
            loss = -(weighted_t*z.float().log_softmax(1)).sum()/w.sum().clamp_min(1e-8)
        if not torch.isfinite(loss):
            raise FloatingPointError('Nonfinite training loss')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            with torch.no_grad():
                for ep, p in zip(ema.parameters(), model.parameters()):
                    ep.lerp_(p, 1-cfg.ema_decay)
                # Keep BN statistics current; do not average integer counters.
                for eb, b in zip(ema.buffers(), model.buffers()):
                    eb.copy_(b)
        total += loss.item()*len(x)
        count += len(x)
    return total/count


def plots(history, out, losses=None, weights=None, ids=None):
    import matplotlib.pyplot as plt
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    df.plot(x='epoch', y=['train_accuracy', 'val_accuracy'], ax=axes[0, 0])
    df.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0, 1])
    df.plot(x='epoch', y=['downweighted', 'relabeled', 'filtered'], ax=axes[1, 0])
    if losses is not None:
        values, w = losses[ids].numpy(), weights[ids].numpy()
        for mask, name in [(w >= .999, 'full weight'), (w < .999, 'downweighted')]:
            if mask.any():
                axes[1, 1].hist(values[mask], bins=40, alpha=.5, label=name)
        axes[1, 1].legend()
        axes[1, 1].set_title('Historical CE against provided labels; not verified noise')
    fig.tight_layout()
    fig.savefig(out/'diagnostics.png', dpi=150)
    plt.close(fig)


def config_signature(cfg):
    values = asdict(cfg)
    for key in ('resume', 'submission', 'submission_root', 'template', 'confirm_positional_ids'):
        values.pop(key)
    # Preserve compatibility with checkpoints written before these optional ablations.
    for key, default in [('freeze_noise_epoch', 0), ('recalibrate_bn', False)]:
        if values[key] == default:
            values.pop(key)
    return hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()


def validate_submission_frame(frame, expected_ids):
    """Validate exact CSV schema, row count/order, and integer class IDs."""
    expected_ids = np.asarray(expected_ids)
    if list(frame.columns) != ['ID', 'target']:
        raise ValueError('Submission columns must be exactly ID,target (no index column).')
    if len(frame) == 0 or len(frame) != len(expected_ids):
        raise ValueError(f'Submission row count mismatch: {len(frame)} vs {len(expected_ids)}.')
    if frame.isna().any().any():
        raise ValueError('Submission contains missing IDs or predictions.')
    for column in ('ID', 'target'):
        if not pd.api.types.is_integer_dtype(frame[column].dtype):
            raise ValueError(f'Submission {column} must contain integers, without fractions or Inf.')
    if frame.ID.duplicated().any() or not np.array_equal(frame.ID.to_numpy(), expected_ids):
        raise ValueError('Submission IDs must match expected row order exactly, without duplicates.')
    if not frame.target.between(0, 99).all():
        raise ValueError('Submission predictions must lie in [0,99].')
    return frame


def preflight_submission(cfg, extra_outputs=(), checkpoint=None, min_free_bytes=512*2**20):
    """Fail before training; test archive is checked by path existence ONLY, never opened."""
    if cfg.smoke or not cfg.submission:
        return {'status': 'skipped', 'reason': 'smoke mode or submission disabled'}
    root = Path(cfg.data_root)
    for relative in ('cifar-100-python/train', 'CIFAR-100-noisy.npz', 'cifar-100-python/test'):
        path = root/relative
        if not path.is_file():
            raise FileNotFoundError(f'Preflight: required competition file missing: {path}')
    if cfg.template:
        template = pd.read_csv(cfg.template)
        if list(template.columns) != ['ID', 'target'] or template.empty:
            raise ValueError('Preflight: template must have ID,target columns and at least one row.')
        if (not pd.api.types.is_integer_dtype(template.ID.dtype) or template.ID.isna().any()
                or template.ID.duplicated().any() or set(template.ID) != set(range(len(template)))):
            raise ValueError('Preflight: this adapter requires unique positional integer template IDs.')
    elif not cfg.confirm_positional_ids:
        raise ValueError('Preflight: provide a template or confirm competition baseline positional IDs.')
    directories = [Path(cfg.output), *map(Path, extra_outputs)]
    destination = Path(cfg.submission_root) if cfg.submission_root else Path(cfg.output)/'submission.csv'
    if destination.suffix.lower() != '.csv' or destination.is_dir():
        raise ValueError(f'Preflight: final CSV must be a file with .csv extension: {destination}')
    protected = [folder/name for folder in directories for name in ('best.pt', 'last.pt', 'summary.json', 'config.json')]
    if destination.resolve() in {p.resolve() for p in protected}:
        raise ValueError('Preflight: submission destination overlaps a checkpoint or metadata file.')
    if checkpoint is not None:
        checkpoint = Path(checkpoint)
        if not checkpoint.is_file():
            raise FileNotFoundError(f'Preflight: checkpoint missing: {checkpoint}. Attach saved outputs in a fresh Kaggle run.')
        if destination.resolve() == checkpoint.resolve():
            raise ValueError('Preflight: preserve the source checkpoint; use a separate CSV destination.')
        if extra_outputs and checkpoint.parent.resolve() in {Path(p).resolve() for p in extra_outputs}:
            raise ValueError('Preflight: keep audit/export output separate from the source checkpoint.')
        # Loading checks architecture/head compatibility; no input images or labels are used.
        saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
        with torch.random.fork_rng(devices=[]):
            probe_model = new_model(torch.device('cpu'))
            probe_model.load_state_dict(saved['model'])
        del saved, probe_model
    directories.append(destination.parent)
    report = dict(status='ok', test_images_opened=False, submission_path=str(destination),
                  checkpoint_verified=str(checkpoint) if checkpoint is not None else None, directories=[])
    probe_frame = pd.DataFrame({'ID': [0, 1, 2], 'target': [0, 99, 7]})
    for directory in dict.fromkeys(directories):
        directory.mkdir(parents=True, exist_ok=True)
        free_bytes = shutil.disk_usage(directory).free
        if free_bytes < min_free_bytes:
            raise OSError(f'Preflight: less than {min_free_bytes/2**20:.0f} MB free at {directory}.')
        fd, name = tempfile.mkstemp(prefix='.submission_probe_', suffix='.tmp', dir=directory)
        os.close(fd)
        probe = Path(name)
        renamed = probe.with_name(probe.name + '.csv')
        try:
            probe_frame.to_csv(probe, index=False)
            probe.replace(renamed)
            validate_submission_frame(pd.read_csv(renamed), np.arange(3))
        finally:
            probe.unlink(missing_ok=True)
            renamed.unlink(missing_ok=True)
        report['directories'].append(dict(path=str(directory), free_mb=round(free_bytes/2**20)))
    (Path(cfg.output)/'preflight.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(f'PREFLIGHT OK: CSV schema and write/rename verified; final path: {destination}; test images not opened.')
    return report


def run(cfg):
    if cfg.method not in ('knn', 'clustering', 'small_loss', 'confidence', 'best'):
        raise ValueError('Unknown experiment method')
    if cfg.smoke:
        cfg.epochs, cfg.feature_epochs, cfg.warmup, cfg.workers = 2, 1, 1, 0
        cfg.batch = min(cfg.batch, 32)
    if cfg.freeze_noise_epoch and (cfg.method not in ('small_loss', 'confidence', 'best')
                                  or cfg.freeze_noise_epoch <= cfg.warmup):
        raise ValueError('Noise snapshot requires a during-training method and an epoch after warmup.')
    preflight_submission(cfg)
    seed_all(cfg.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    out = Path(cfg.output)
    out.mkdir(parents=True, exist_ok=True)
    images, original = load_training(cfg.data_root)
    train_ids, val_ids = split_indices(len(images), cfg)
    split_hash = hashlib.sha256(train_ids.tobytes()+val_ids.tobytes()).hexdigest()
    signature = config_signature(cfg)
    checkpoint_path = out/'last.pt'
    resumed = None
    if checkpoint_path.exists():
        if not cfg.resume:
            raise FileExistsError('Use a new output directory for a new experiment.')
        resumed = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        if resumed['signature'] != signature or resumed['split_hash'] != split_hash:
            raise ValueError('Resume configuration/split mismatch. Use a new output directory.')
    np.savez(out/'split.npz', train_ids=train_ids, val_ids=val_ids)
    (out/'config.json').write_text(json.dumps(asdict(cfg), indent=2), encoding='utf-8')
    labels, active = original.copy(), train_ids.copy()
    prep_start = time.perf_counter()
    prep_seconds = 0.
    if resumed is not None:
        labels, active = resumed['labels'], resumed['active']
        prep_seconds = resumed['prep_seconds']
    elif cfg.method in ('knn', 'clustering'):
        # Feature teacher sees only the training partition. Never uses validation labels.
        teacher = new_model(device)
        opt = torch.optim.SGD(teacher.parameters(), lr=cfg.lr, momentum=.9, weight_decay=cfg.weight_decay)
        scaler = torch.amp.GradScaler(device.type, enabled=device.type == 'cuda')
        targets = F.one_hot(torch.as_tensor(original), 100).float()
        weights = torch.ones(len(original))
        feature_loader = loader(images, original, train_ids, cfg, True)
        for epoch in range(cfg.feature_epochs):
            for group in opt.param_groups:
                group['lr'] = cfg.lr*.5*(1+math.cos(math.pi*epoch/cfg.feature_epochs))
            loss = train_epoch(teacher, feature_loader, opt, scaler, targets, weights, device, cfg)
            print(f'Feature teacher {epoch+1}/{cfg.feature_epochs}: loss={loss:.4f}', flush=True)
        _, _, features = inspect_training(teacher, loader(images, original, train_ids, cfg), len(images), device, True)
        labels, active, suspect = fixed_preprocessing(features, original, train_ids, cfg)
        np.savez(out/'fixed_preprocessing.npz', ids=train_ids, labels=labels[train_ids], keep=np.isin(train_ids, active), suspect=suspect)
        torch.save(teacher.state_dict(), out/'feature_teacher.pt')
        del teacher, features
        prep_seconds = time.perf_counter()-prep_start
    # Fresh initialization after preprocessing; no teacher weights transferred.
    seed_all(cfg.seed+100)
    model = new_model(device)
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters():
        p.requires_grad_(False)
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=.9, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == 'cuda')
    histories = History(len(images))
    history, best, best_epoch, start_epoch, elapsed_before = [], -1., 0, 0, 0.
    frozen_noise = None
    if resumed is not None:
        model.load_state_dict(resumed['model'])
        ema.load_state_dict(resumed['ema'])
        optimizer.load_state_dict(resumed['optimizer'])
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(device)
        scaler.load_state_dict(resumed['scaler'])
        histories.__dict__.update(resumed['sample_history'])
        frozen_noise = resumed.get('frozen_noise')
        history, best, best_epoch = resumed['history'], resumed['best'], resumed['best_epoch']
        start_epoch, elapsed_before = resumed['epoch']+1, resumed['elapsed']
        random.setstate(resumed['python_rng'])
        np.random.set_state(resumed['numpy_rng'])
        torch.set_rng_state(resumed['torch_rng'])
        if device.type == 'cuda' and resumed['cuda_rng'] is not None:
            torch.cuda.set_rng_state_all(resumed['cuda_rng'])
    train_loader = loader(images, labels, active, cfg, True)
    probe_loader = loader(images, original, train_ids, cfg)
    val_loader = loader(images, original, val_ids, cfg)
    started = time.perf_counter()
    fixed_count = int(np.sum(labels[train_ids] != original[train_ids]))
    filtered = len(train_ids)-len(active)
    if frozen_noise is None:
        targets, weights, changed = dynamic_targets(labels, train_ids, histories, cfg, max(0, start_epoch-1))
    else:
        targets, weights, changed = frozen_noise['targets'], frozen_noise['weights'], frozen_noise['changed']
    # Before methods never enter the dynamic noise-estimation branch.
    for epoch in range(start_epoch, cfg.epochs):
        epoch_start = time.perf_counter()
        lr = cfg.lr*min(1., (epoch+1)/5)*.5*(1+math.cos(math.pi*epoch/cfg.epochs))
        for group in optimizer.param_groups:
            group['lr'] = lr
        if cfg.method in ('small_loss', 'confidence', 'best'):
            if frozen_noise is None:
                probs, losses, _ = inspect_training(ema, probe_loader, len(images), device)
                histories.update(probs, losses, train_ids, cfg)
            targets, weights, changed, frozen_noise = noise_targets(labels, train_ids, histories, cfg, epoch, frozen_noise)
        else:
            targets = F.one_hot(torch.as_tensor(labels), 100).float()
            weights, changed = torch.ones(len(labels)), torch.zeros(len(labels), dtype=torch.bool)
        train_loss = train_epoch(model, train_loader, optimizer, scaler, targets, weights, device, cfg, ema)
        if cfg.recalibrate_bn:
            calibrate_batchnorm(model, probe_loader, device)
            calibrate_batchnorm(ema, probe_loader, device)
        # Compare raw and EMA checkpoints using the same noisy validation criterion.
        raw_accuracy, raw_loss = evaluate(model, val_loader, device)
        ema_accuracy, ema_loss = evaluate(ema, val_loader, device)
        chosen, accuracy, val_loss = (ema, ema_accuracy, ema_loss) if ema_accuracy >= raw_accuracy else (model, raw_accuracy, raw_loss)
        train_accuracy, observed_loss = evaluate(chosen, probe_loader, device)
        if accuracy > best:
            best, best_epoch = accuracy, epoch+1
            torch.save({'model': chosen.state_dict(), 'epoch': best_epoch, 'accuracy': best,
                        'split_hash': split_hash, 'inference_tta': False,
                        'source': 'ema' if chosen is ema else 'raw'}, out/'best.pt')
        row = dict(epoch=epoch+1, train_accuracy=train_accuracy, val_accuracy=accuracy,
                   train_loss=observed_loss, objective_loss=train_loss, val_loss=val_loss,
                   raw_val_accuracy=raw_accuracy, ema_val_accuracy=ema_accuracy,
                   downweighted=int((weights[active] < .999).sum()), relabeled=int(changed[active].sum())+fixed_count,
                   filtered=filtered, noise_frozen=frozen_noise is not None,
                   mean_weight=float(weights[active].mean()),
                   weight_below_half=int((weights[active] < .5).sum()),
                   lr=lr, seconds=time.perf_counter()-epoch_start)
        history.append(row)
        pd.DataFrame(history).to_csv(out/'history.csv', index=False)
        pd.DataFrame({'original_id': train_ids, 'weight': weights[train_ids].numpy(),
                      'soft_corrected': changed[train_ids].numpy()}).to_csv(out/'latest_decisions.csv', index=False)
        (out/'decisions').mkdir(exist_ok=True)
        np.savez_compressed(out/'decisions'/f'epoch_{epoch+1:03d}.npz', ids=train_ids,
                            weight=weights[train_ids].numpy(), soft_corrected=changed[train_ids].numpy(),
                            included=np.isin(train_ids, active))
        if cfg.gradient_signal and cfg.method in ('small_loss', 'confidence', 'best'):
            # CE logit gradient p-onehot(y); last-layer Frobenius norm = ||p-y|| ||h||.
            g = histories.probs[train_ids]-F.one_hot(torch.as_tensor(original[train_ids]), 100)
            np.save(out/'logit_gradient_norm.npy', g.norm(dim=1).numpy())
        elapsed = elapsed_before + time.perf_counter()-started
        state = dict(signature=signature, split_hash=split_hash, model=model.state_dict(), ema=ema.state_dict(),
                     frozen_noise=frozen_noise,
                     optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), sample_history=histories.__dict__,
                     history=history, best=best, best_epoch=best_epoch, epoch=epoch, labels=labels, active=active,
                     prep_seconds=prep_seconds, elapsed=elapsed, python_rng=random.getstate(), numpy_rng=np.random.get_state(),
                     torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if device.type == 'cuda' else None)
        temporary = out/'last.tmp'
        torch.save(state, temporary)
        temporary.replace(checkpoint_path)
        print(row, flush=True)
    selected = torch.load(out/'best.pt', map_location=device, weights_only=True)
    model.load_state_dict(selected['model'])
    best_loss = evaluate(model, val_loader, device)[1]
    summary = dict(method=cfg.method, best_validation_accuracy=best, validation_loss=best_loss, best_epoch=best_epoch,
                   seed=cfg.seed, split_hash=split_hash, architecture='CIFAR ResNet18', optimizer='SGD momentum 0.9',
                   scheduler='5-epoch LR warmup + cosine', seconds=elapsed_before+time.perf_counter()-started,
                   preprocessing_seconds=prep_seconds, filtered=filtered, fixed_relabels=fixed_count,
                   last_dynamic_relabels=history[-1]['relabeled']-fixed_count,
                   last_downweighted=history[-1]['downweighted'], smoke=cfg.smoke,
                   freeze_noise_epoch=cfg.freeze_noise_epoch, recalibrate_bn=cfg.recalibrate_bn,
                   peak_gpu_mb=torch.cuda.max_memory_allocated()/2**20 if device.type == 'cuda' else 0,
                   submission_path=None, kaggle_notebook_url=None, kaggle_score=None)
    counts = pd.DataFrame({'original': np.bincount(original[train_ids], minlength=100),
                           'final_fixed': np.bincount(labels[active], minlength=100)})
    counts.to_csv(out/'class_balance.csv', index_label='class')
    if cfg.method in ('knn', 'clustering'):
        # Diagnostics only: no label/subset changes during final-model training.
        _, diagnostic_losses, _ = inspect_training(model, probe_loader, len(images), device)
    else:
        diagnostic_losses = histories.loss
    # Persist training evidence even if later inference or plotting fails.
    (out/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    if cfg.submission and not cfg.smoke:
        export_best_submission(model, cfg, summary)
    elif cfg.submission:
        print('Smoke mode: submission skipped; competition test data was not opened.')
    plots(history, out, diagnostic_losses, weights, train_ids)
    print(json.dumps(summary, indent=2))
    return model, summary


def audit_checkpoint(cfg, checkpoint_path, output):
    """Four validation-only inference ablations; original checkpoint is never overwritten."""
    checkpoint_path, out = Path(checkpoint_path), Path(output)
    if cfg.smoke:
        raise ValueError('Audit full checkpoints with the original full validation split, not smoke mode.')
    if out.resolve() == checkpoint_path.parent.resolve():
        raise ValueError('Use a separate audit output directory to preserve the original checkpoint.')
    preflight_submission(cfg, extra_outputs=[out], checkpoint=checkpoint_path)
    seed_all(cfg.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    images, labels = load_training(cfg.data_root)
    train_ids, val_ids = split_indices(len(images), cfg)
    split_hash = hashlib.sha256(train_ids.tobytes()+val_ids.tobytes()).hexdigest()
    if checkpoint.get('split_hash', split_hash) != split_hash:
        raise ValueError('Checkpoint uses a different split.')
    if 'split_hash' not in checkpoint:
        print('Legacy checkpoint: ensure seed and val_fraction match its original run.')
    model = new_model(device)
    model.load_state_dict(checkpoint['model'])
    val_loader = loader(images, labels, val_ids, cfg)
    train_loader = loader(images, labels, train_ids, cfg)
    out.mkdir(parents=True, exist_ok=True)
    rows, best_key, best_state, best_row = [], None, None, None
    for calibrated in (False, True):
        if calibrated:
            calibrate_batchnorm(model, train_loader, device)
        for tta in (False, True):
            accuracy, loss = evaluate(model, val_loader, device, tta=tta)
            row = dict(bn_recalibrated=calibrated, tta=tta, validation_accuracy=accuracy, validation_loss=loss)
            rows.append(row)
            # Accuracy first, validation loss as tie breaker; never any test feedback.
            key = (accuracy, -loss)
            if best_key is None or key > best_key:
                best_key, best_row = key, row.copy()
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    audit_best = dict(model=best_state, epoch=checkpoint.get('epoch'), accuracy=best_row['validation_accuracy'],
                      validation_loss=best_row['validation_loss'],
                      split_hash=split_hash, source=str(checkpoint_path), inference_tta=best_row['tta'],
                      bn_recalibrated=best_row['bn_recalibrated'])
    torch.save(audit_best, out/'best.pt')
    table = pd.DataFrame(rows)
    table.to_csv(out/'inference_ablation.csv', index=False)
    (out/'audit.json').write_text(json.dumps(dict(best=best_row, rows=rows, source=str(checkpoint_path),
                                  seed=cfg.seed, val_fraction=cfg.val_fraction), indent=2), encoding='utf-8')
    model.load_state_dict(best_state)
    print(table.to_string(index=False))
    print(f'Selected audit checkpoint: {out / "best.pt"}; inference TTA={best_row["tta"]}')
    return model, table, audit_best


def run_with_final_audit(cfg, audit_output):
    """Train once, select BN/TTA on allowed validation, then infer test exactly once."""
    requested_submission = cfg.submission
    training_output = cfg.output
    if Path(training_output).resolve() == Path(audit_output).resolve():
        raise ValueError('Final audit output must differ from training output.')
    preflight_submission(cfg, extra_outputs=[audit_output])
    try:
        cfg.submission = False
        model, summary = run(cfg)
    finally:
        cfg.submission = requested_submission
    if cfg.smoke:
        print('Smoke mode: final audit and submission skipped.')
        return model, summary
    model, _, selected = audit_checkpoint(cfg, Path(training_output)/'best.pt', audit_output)
    cfg.output = str(audit_output)
    cfg.tta = selected['inference_tta']
    summary = dict(summary, training_output=str(training_output),
                   training_best_validation_accuracy=summary['best_validation_accuracy'],
                   best_validation_accuracy=selected['accuracy'], validation_loss=selected['validation_loss'],
                   bn_recalibrated=selected['bn_recalibrated'], inference_tta=selected['inference_tta'],
                   source_checkpoint=str(Path(training_output)/'best.pt'), submission_path=None)
    (Path(cfg.output)/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    if requested_submission:
        export_best_submission(model, cfg, summary)
    print(json.dumps(summary, indent=2))
    return model, summary


def export_best_submission(model, cfg, summary=None):
    """Always use the active run's best checkpoint; export both per-run and root CSVs."""
    if cfg.smoke:
        raise ValueError('Smoke models cannot produce competition submissions.')
    out = Path(cfg.output)
    checkpoint_path = out/'best.pt'
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    model.load_state_dict(checkpoint['model'])
    cfg.tta = checkpoint.get('inference_tta', cfg.tta)
    experiment_path = make_submission(model, cfg, next(model.parameters()).device)
    frame = pd.read_csv(experiment_path)
    expected_ids = pd.read_csv(cfg.template).ID.to_numpy() if cfg.template else np.arange(len(frame))
    validate_submission_frame(frame, expected_ids)
    destination = Path(cfg.submission_root) if cfg.submission_root else experiment_path
    if destination.resolve() != experiment_path.resolve():
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + '.tmp')
        shutil.copyfile(experiment_path, temporary)
        temporary.replace(destination)
    if summary is None:
        summary_file = out/'summary.json'
        summary = json.loads(summary_file.read_text(encoding='utf-8')) if summary_file.exists() else {}
    summary.update(submission_path=str(destination), experiment_submission_path=str(experiment_path),
                   submission_checkpoint=str(checkpoint_path), submission_tta=bool(cfg.tta),
                   submission_rows=len(frame),
                   submission_sha256=hashlib.sha256(destination.read_bytes()).hexdigest())
    temporary_summary = out/'summary.json.tmp'
    temporary_summary.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    temporary_summary.replace(out/'summary.json')
    print(f'Submission ready: {destination}')
    print(f'Experiment copy: {experiment_path}; checkpoint: {checkpoint_path}; TTA={cfg.tta}')
    return destination


def export_checkpoint(cfg, checkpoint_path, output):
    """Recover/generate a submission from own saved weights, without any training or audit."""
    if cfg.smoke:
        raise ValueError('Cannot export a competition submission in smoke mode.')
    checkpoint_path, out = Path(checkpoint_path), Path(output)
    if out.resolve() == checkpoint_path.parent.resolve():
        raise ValueError('Use a separate writable export directory; preserve the source checkpoint.')
    requested_submission = cfg.submission
    try:
        # Explicit export is requested even when automatic export was disabled.
        cfg.submission = True
        preflight_submission(cfg, extra_outputs=[out], checkpoint=checkpoint_path)
    finally:
        cfg.submission = requested_submission
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = new_model(device)
    model.load_state_dict(checkpoint['model'])
    out.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint, out/'best.pt')
    cfg.output = str(out)
    summary = dict(method='checkpoint_export', source_checkpoint=str(checkpoint_path), submission_path=None)
    export_best_submission(model, cfg, summary)
    return model, summary


def submission_frame(predictions, cfg):
    n = len(predictions)
    if cfg.template:
        template = pd.read_csv(cfg.template)
        if list(template.columns) != ['ID', 'target'] or len(template) != n or template.ID.duplicated().any():
            raise ValueError('Template must have ID,target columns and exactly N unique IDs.')
        # This adapter maps row i of the competition pickle to numeric ID i.
        ids = template.ID.to_numpy()
        if set(ids.tolist()) != set(range(n)):
            raise ValueError('Non-positional IDs: provide a verified row-to-ID adapter before inference.')
        result = template.copy()
        result['target'] = np.asarray(predictions)[ids.astype(int)]
    else:
        if not cfg.confirm_positional_ids:
            raise ValueError('Provide sample submission or explicitly confirm baseline IDs are range(N).')
        result = pd.DataFrame({'ID': np.arange(n), 'target': predictions})
    validate_submission_frame(result, result.ID.to_numpy() if cfg.template else np.arange(n))
    return result


@torch.inference_mode()
def make_submission(model, cfg, device):
    """Only test-data entry point: frozen-model forward inference, no metrics/tuning."""
    # Verify mapping authorization before opening any test archive.
    if not cfg.template and not cfg.confirm_positional_ids:
        raise ValueError('Set template or confirm_positional_ids based on the competition baseline.')
    with (Path(cfg.data_root)/'cifar-100-python/test').open('rb') as f:
        data = pickle.load(f, encoding='bytes')[b'data']
    if data.dtype != np.uint8 or data.ndim != 2 or data.shape[1] != 3072:
        raise ValueError('Unsupported test image storage')
    images = data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    batches = loader(images, np.zeros(len(images), dtype=np.int64), np.arange(len(images)), cfg)
    predictions = []
    model.eval()
    for x, _, _ in batches:
        x = x.to(device, non_blocking=True)
        with torch.autocast(device.type, enabled=device.type == 'cuda'):
            probs = model(x).float().softmax(1)
            if cfg.tta:
                probs = (probs+model(x.flip(-1)).float().softmax(1))/2
        predictions.extend(probs.argmax(1).cpu().tolist())
    result = submission_frame(predictions, cfg)
    path = Path(cfg.output)/'submission.csv'
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    result.to_csv(temporary, index=False)
    validate_submission_frame(pd.read_csv(temporary), result.ID.to_numpy())
    temporary.replace(path)
    return path


In [ ]:
cfg = Config(**{'method': 'knn', 'output': '/kaggle/working/01_before_knn', 'submission': True, 'submission_root': '/kaggle/working/submission.csv', 'confirm_positional_ids': True})
cfg.smoke = False
# Set cfg.data_root if the Kaggle mount differs.
# Use a distinct output folder for each ablation.


In [ ]:
preflight_report = preflight_submission(cfg)


In [ ]:
model, summary = run(cfg)


In [ ]:
from IPython.display import Image, display
display(pd.read_csv(Path(cfg.output) / "history.csv"))
display(Image(filename=str(Path(cfg.output) / "diagnostics.png")))


## Submission



In [ ]:
if cfg.smoke:
    print("Smoke run: no competition submission generated.")
else:
    submission_path = Path(summary["submission_path"])
    frame = pd.read_csv(submission_path)
    validate_submission_frame(frame, np.arange(summary["submission_rows"]))
    assert hashlib.sha256(submission_path.read_bytes()).hexdigest() == summary["submission_sha256"]
    print(f"Ready to submit: {submission_path}; rows={len(frame)}")
    display(frame.head())
